## 1. 28通道电容值

In [5]:
import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

### 1.1 数据准备

In [11]:
# 1. 加载数据 (假设你在 MATLAB 中已经用 table2array 存为了 dataArray)
mat_data = sio.loadmat('data_array.mat')
data = mat_data['dataArray'] # 提取 9000x31 的矩阵

# 2. 分离特征 (X) 和 标签 (Y)
# 按照你的表格列顺序（Python 索引从 0 开始）：
# 0: Sigma, 1: Thickness (目标), 2: Freq, 3~30: Cap_Ch1~28
Sigma = data[:, 0].reshape(-1, 1)
Thickness = data[:, 1]           # 这是我们要预测的目标 Y
Freq = data[:, 2].reshape(-1, 1)
Caps = data[:, 3:31]             # 28个电容通道数据
Caps = data[:, 3:31] * 1e12  # 人为放大电容的数值量级

# 拼接所有输入特征构造成 9000 x 30 的 X 矩阵
X = np.hstack((Sigma, Freq, Caps))
y = Thickness

# 3. 划分训练集和测试集 (80%训练, 20%测试)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. 数据标准化 (极其重要！因为电容值是 1e-11 级别，而频率可能是 1e3 级别)
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

### 1.2 随机森林模型

In [12]:
print("--- 正在训练随机森林模型 ---")
# 构建并训练模型
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# 预测
y_pred_rf = rf_model.predict(X_test_scaled)

# 评估
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"随机森林 - MSE (均方误差): {mse_rf:.4f}")
print(f"随机森林 - R2 Score (决定系数): {r2_rf:.4f}")

--- 正在训练随机森林模型 ---
随机森林 - MSE (均方误差): 48.1415
随机森林 - R2 Score (决定系数): 0.1313


### 1.3 小型MLP模型

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("\n--- 正在训练小型神经网络模型 ---")

# 转换为 PyTorch 的 Tensor
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# 构建 DataLoader
dataset = TensorDataset(X_train_t, y_train_t)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# 定义小型多层感知机 (MLP)
class ECT_Net(nn.Module):
    def __init__(self):
        super(ECT_Net, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 128),   # 输入层: 30个特征
            nn.ReLU(),
            nn.Linear(128, 64),   # 隐藏层
            nn.ReLU(),
            nn.Linear(64, 32),    # 隐藏层
            nn.ReLU(),
            nn.Linear(32, 1)      # 输出层: 1个预测值 (Thickness)
        )

    def forward(self, x):
        return self.net(x)

model = ECT_Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练网络
epochs = 150
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch+1) % 30 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.4f}")

# 测试网络
model.eval()
with torch.no_grad():
    y_pred_nn = model(X_test_t)
    mse_nn = mean_squared_error(y_test, y_pred_nn.numpy())
    r2_nn = r2_score(y_test, y_pred_nn.numpy())
    print(f"神经网络 - MSE (均方误差): {mse_nn:.4f}")
    print(f"神经网络 - R2 Score (决定系数): {r2_nn:.4f}")


--- 正在训练小型神经网络模型 ---
Epoch [30/150], Loss: 49.5793
Epoch [60/150], Loss: 49.1815
Epoch [90/150], Loss: 49.0259
Epoch [120/150], Loss: 49.0071
Epoch [150/150], Loss: 48.9995
神经网络 - MSE (均方误差): 48.7588
神经网络 - R2 Score (决定系数): 0.1201


### 1.4 强化版MLP

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print("\n--- 正在训练强化版神经网络 (Pro Version) ---")

# 1. 【新增】对目标变量 Y 也进行标准化
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

# 2. 转换为 PyTorch Tensor
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32) # 注意这里用了 scaled 的 y
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32)

# 构建 DataLoader
dataset = TensorDataset(X_train_t, y_train_t)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True) # Batch size 稍微调大有助于 BN 层的稳定

# 3. 【改进】带有 Batch Normalization 和 Dropout 的强健网络
class ECT_Net_Pro(nn.Module):
    def __init__(self):
        super(ECT_Net_Pro, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 256),       # 加宽网络
            nn.BatchNorm1d(256),      # 【新增】批归一化，加速收敛，稳定训练
            nn.ReLU(),
            nn.Dropout(0.1),          # 【新增】防止过拟合
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, 1)          # 输出层不需要激活和归一化
        )

    def forward(self, x):
        return self.net(x)

model = ECT_Net_Pro()
criterion = nn.MSELoss()
# 使用 weight_decay 加入 L2 正则化
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5) 

# 4. 【新增】学习率调度器：如果 10 个 Epoch loss 没降，学习率减半
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

# 5. 训练网络
epochs = 200
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    
    # 用训练集的平均 loss 更新学习率（更严谨的做法是用验证集 loss）
    scheduler.step(avg_loss) 
    
    if (epoch+1) % 20 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")

# 6. 测试与反归一化评估
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t)
    
    # 【新增】将预测结果反归一化回真实的物理厚度值
    y_pred_real = scaler_y.inverse_transform(y_pred_scaled.numpy())
    
    # 计算最终指标（和真实的 y_test 比较）
    mse_nn = mean_squared_error(y_test, y_pred_real)
    r2_nn = r2_score(y_test, y_pred_real)
    
    print(f"\n强化版神经网络 - MSE (均方误差): {mse_nn:.4f}")
    print(f"强化版神经网络 - R2 Score (决定系数): {r2_nn:.4f}")


--- 正在训练强化版神经网络 (Pro Version) ---


e:\Anaconda3\envs\torch_gpu\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [20/200], Loss: 0.8972, LR: 0.005000
Epoch [40/200], Loss: 0.8930, LR: 0.005000
Epoch [60/200], Loss: 0.8882, LR: 0.002500
Epoch [80/200], Loss: 0.8871, LR: 0.001250
Epoch [100/200], Loss: 0.8829, LR: 0.000313
Epoch [120/200], Loss: 0.8848, LR: 0.000156
Epoch [140/200], Loss: 0.8900, LR: 0.000078
Epoch [160/200], Loss: 0.8850, LR: 0.000039
Epoch [180/200], Loss: 0.8870, LR: 0.000010
Epoch [200/200], Loss: 0.8859, LR: 0.000002

强化版神经网络 - MSE (均方误差): 48.4002
强化版神经网络 - R2 Score (决定系数): 0.1266


## 2. 电导+电容+阻抗角组合特征

### 2.1 数据准备

In [18]:
import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 加载新的数据 (假设新矩阵是 9000 x 87)
mat_data = sio.loadmat('data_array.mat')
data = mat_data['dataArray_cap'] 

# 2. 分离特征 (X) 和 标签 (Y)
# 索引按照列顺序：
Sigma = data[:, 0].reshape(-1, 1)        # 第1列: 介质电导率
Thickness = data[:, 1]                   # 第2列: 环流厚度 (目标Y)
Freq = data[:, 2].reshape(-1, 1)         # 第3列: 激励频率

# 新的 28x3 = 84 个通道数据切片 (请核对你的 MATLAB 实际列数排布)
Caps = data[:, 3:31]                     # 第4~31列: 28个电容值
Conds = data[:, 31:59]                   # 第32~59列: 28个电导测量值 (新增)
Angles = data[:, 59:87]                  # 第60~87列: 28个阻抗角测量值 (新增)

# 拼接所有输入特征构造成 9000 x 86 的 X 矩阵
# 现在的特征维度为：1 + 1 + 28 + 28 + 28 = 86 维
X = np.hstack((Sigma, Freq, Caps, Conds, Angles))
y = Thickness

# 3. 划分数据集和标准化 (保持不变，但 StandardScaler 现在处理 86 维数据)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

### 2.2 Pro_V2

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 将数据转为 Tensor
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32)

dataset = TensorDataset(X_train_t, y_train_t)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

class ECT_Net_Pro_V2(nn.Module):
    def __init__(self):
        super(ECT_Net_Pro_V2, self).__init__()
        self.net = nn.Sequential(
            # 【关键修改】：输入层维度改为 86，神经元增加到 512
            nn.Linear(86, 512),       
            nn.BatchNorm1d(512),      
            nn.ReLU(),
            nn.Dropout(0.1),          
            
            # 第二层加宽到 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            # 第三层
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            # 输出层保持不变，依然预测 1 个厚度值
            nn.Linear(64, 1)          
        )

    def forward(self, x):
        return self.net(x)

# 实例化新网络
model = ECT_Net_Pro_V2()
criterion = nn.MSELoss()

# 学习率可以稍微回调一点，因为网络变深变宽了
optimizer = optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-5) 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

# ----- 后续的训练循环和测试代码 (epochs 的 for 循环等) 与之前完全一样，无需修改 -----

# 5. 训练网络
epochs = 200
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    
    # 用训练集的平均 loss 更新学习率（更严谨的做法是用验证集 loss）
    scheduler.step(avg_loss) 
    
    if (epoch+1) % 20 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")

# 6. 测试与反归一化评估
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t)
    
    # 【新增】将预测结果反归一化回真实的物理厚度值
    y_pred_real = scaler_y.inverse_transform(y_pred_scaled.numpy())
    
    # 计算最终指标（和真实的 y_test 比较）
    mse_nn = mean_squared_error(y_test, y_pred_real)
    r2_nn = r2_score(y_test, y_pred_real)
    
    print(f"\n强化版神经网络 - MSE (均方误差): {mse_nn:.4f}")
    print(f"强化版神经网络 - R2 Score (决定系数): {r2_nn:.4f}")

Epoch [20/200], Loss: 0.8993, LR: 0.003000
Epoch [40/200], Loss: 0.8904, LR: 0.003000
Epoch [60/200], Loss: 0.8884, LR: 0.001500
Epoch [80/200], Loss: 0.8844, LR: 0.000750
Epoch [100/200], Loss: 0.8886, LR: 0.000375
Epoch [120/200], Loss: 0.8867, LR: 0.000188
Epoch [140/200], Loss: 0.8853, LR: 0.000047
Epoch [160/200], Loss: 0.8872, LR: 0.000023
Epoch [180/200], Loss: 0.8830, LR: 0.000006
Epoch [200/200], Loss: 0.8847, LR: 0.000003

强化版神经网络 - MSE (均方误差): 48.3560
强化版神经网络 - R2 Score (决定系数): 0.1274


### 2.3 Pro_V3

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 假设前面 X_train_scaled 等数据的加载和预处理(86维输入)保持不变
# 转换为 PyTorch Tensor (这里代码与之前相同)
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32)

dataset = TensorDataset(X_train_t, y_train_t)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# ==========================================
# 🌟 核心：基于注意力机制的多模态传感器网络
# ==========================================
class ECT_Attention_Net(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2):
        super(ECT_Attention_Net, self).__init__()
        
        # 1. 传感器通道特征嵌入 (将 3个物理量 升维到 d_model)
        self.sensor_proj = nn.Linear(3, d_model)
        
        # 2. Transformer 编码器层 (引入多头自注意力机制)
        # batch_first=True 表示输入形状为 (Batch, Seq_len, Features)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 2, 
            dropout=0.1, 
            batch_first=True 
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. 全局特征网络 (单独处理 Sigma 和 Freq)
        self.global_proj = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU()
        )
        
        # 4. 最终的回归层
        # Transformer 展平后的维度: 28个通道 * d_model
        # 加上全局特征的 16 维
        in_features = 28 * d_model + 16
        self.regressor = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, 1) # 输出厚度
        )

    def forward(self, x):
        # x 的形状是 (Batch, 86)
        
        # 1. 拆分全局特征 (前2列: Sigma, Freq)
        x_global = x[:, 0:2] 
        
        # 2. 拆分并重组空间传感器特征
        # 第2~29列是电容, 30~57是电导, 58~85是阻抗角
        caps = x[:, 2:30].unsqueeze(-1)     # (Batch, 28, 1)
        conds = x[:, 30:58].unsqueeze(-1)   # (Batch, 28, 1)
        angles = x[:, 58:86].unsqueeze(-1)  # (Batch, 28, 1)
        
        # 将三个物理量拼接在一起，形成 (Batch, 28通道, 3个特征) 的序列结构
        x_sensor = torch.cat([caps, conds, angles], dim=-1) 
        
        # 3. 注意力机制前向传播
        # 线性映射: (Batch, 28, 3) -> (Batch, 28, d_model)
        seq = self.sensor_proj(x_sensor) 
        
        # 提取空间通道的注意力: (Batch, 28, d_model)
        att_out = self.transformer(seq)  
        
        # 展平注意力输出: (Batch, 28 * d_model)
        att_flat = att_out.flatten(start_dim=1) 
        
        # 4. 融合并预测
        glob_feat = self.global_proj(x_global)                 # (Batch, 16)
        combined = torch.cat([att_flat, glob_feat], dim=1)     # (Batch, 28*d_model + 16)
        
        out = self.regressor(combined)
        return out

# ==========================================
# 实例化与训练
# ==========================================
model = ECT_Attention_Net(d_model=64, nhead=4, num_layers=2)
criterion = nn.MSELoss()

# 使用稍微小一点的学习率，Transformer对学习率比较敏感
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15, verbose=True)

print("\n--- 正在训练基于多头注意力机制的模型 ---")
epochs = 200
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        # 梯度裁剪，防止 Transformer 训练初期梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    scheduler.step(avg_loss)
    
    if (epoch+1) % 20 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch[{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")

# 测试评估
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t)
    y_pred_real = scaler_y.inverse_transform(y_pred_scaled.numpy())
    
    mse_nn = mean_squared_error(y_test, y_pred_real)
    r2_nn = r2_score(y_test, y_pred_real)
    
    print(f"\nAttention网络 - MSE: {mse_nn:.4f}")
    print(f"Attention网络 - R2 Score: {r2_nn:.4f}")

e:\Anaconda3\envs\torch_gpu\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



--- 正在训练基于多头注意力机制的模型 ---
Epoch[20/200], Loss: 0.8926, LR: 0.001000
Epoch[40/200], Loss: 0.8888, LR: 0.000500
Epoch[60/200], Loss: 0.8882, LR: 0.000250
Epoch[80/200], Loss: 0.8906, LR: 0.000125
Epoch[100/200], Loss: 0.8874, LR: 0.000063
Epoch[120/200], Loss: 0.8847, LR: 0.000031
Epoch[140/200], Loss: 0.8864, LR: 0.000016
Epoch[160/200], Loss: 0.8867, LR: 0.000008
Epoch[180/200], Loss: 0.8847, LR: 0.000004
Epoch[200/200], Loss: 0.8899, LR: 0.000002

Attention网络 - MSE: 48.3086
Attention网络 - R2 Score: 0.1283
